In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Бинарная классификация: LightAutoML vs XGBoost

In [ ]:
# Загрузка данных
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')
TARGET_NAME = 'TARGET'

# Разделение на признаки и целевую переменную
X = df_train.drop(columns=[TARGET_NAME, 'ID'])
y = df_train[TARGET_NAME]

# Стратифицированное разбиение на train/val
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=RANDOM_STATE
)

Train size: 60816
Val size: 15204


## LightAutoML

Обучаем две конфигурации AutoML:
- **Конфиг 1** — стандартные настройки
- **Конфиг 2** — только градиентный бустинг (LightGBM + CatBoost)

In [ ]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

task = Task('binary')
roles = {'target': TARGET_NAME}

# Конфиг 1: стандартные настройки
automl_1 = TabularAutoML(
    task=task, 
    timeout=600,
    cpu_limit=4, 
    reader_params={'n_jobs': 4, 'cv': 5, 'random_state': RANDOM_STATE}
)

train_data = pd.concat([X_train, y_train], axis=1)
oof_pred_1 = automl_1.fit_predict(train_data, roles=roles, verbose=1)
val_pred_1 = automl_1.predict(X_val)
score_1 = roc_auc_score(y_val, val_pred_1.data[:, 0])

[19:24:07] Stdout logging level is INFO.
[19:24:07] Task: binary

[19:24:07] Start automl preset with listed constraints:
[19:24:07] - time: 600.00 seconds
[19:24:07] - CPU: 4 cores
[19:24:07] - memory: 16 GB

[19:24:07] Train data shape: (60816, 371)

[19:24:21] Layer 1 train process start. Time left 585.93 secs
[19:24:23] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[19:24:35] Fitting Lvl_0_Pipe_0_Mod_0_LinearL2 finished. score = 0.7914950397219301
[19:24:35] Lvl_0_Pipe_0_Mod_0_LinearL2 fitting and predicting completed
[19:24:35] Time left 571.30 secs

[19:24:37] Selector_LightGBM fitting and predicting completed
[19:24:39] Start fitting Lvl_0_Pipe_1_Mod_0_LightGBM ...
[19:24:48] Fitting Lvl_0_Pipe_1_Mod_0_LightGBM finished. score = 0.8310780537385636
[19:24:48] Lvl_0_Pipe_1_Mod_0_LightGBM fitting and predicting completed
[19:24:48] Start hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ... Time budget is 93.97 secs


Optimization Progress:  36%|███▌      | 36/101 [01:36<02:53,  2.67s/it, best_trial=16, best_value=0.847]

[19:26:24] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM completed
[19:26:24] Start fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM ...


[19:26:30] Fitting Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM finished. score = 0.8324470453723593
[19:26:30] Lvl_0_Pipe_1_Mod_1_Tuned_LightGBM fitting and predicting completed
[19:26:30] Start fitting Lvl_0_Pipe_1_Mod_2_CatBoost ...
[19:26:51] Fitting Lvl_0_Pipe_1_Mod_2_CatBoost finished. score = 0.8327864639035865
[19:26:51] Lvl_0_Pipe_1_Mod_2_CatBoost fitting and predicting completed
[19:26:51] Start hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ... Time budget is 235.22 secs


Optimization Progress:  33%|███▎      | 33/101 [04:00<08:16,  7.30s/it, best_trial=29, best_value=0.846]

[19:30:52] Hyperparameters optimization for Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost completed
[19:30:52] Start fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost ...


[19:32:02] Fitting Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost finished. score = 0.8367088399528485
[19:32:02] Lvl_0_Pipe_1_Mod_3_Tuned_CatBoost fitting and predicting completed
[19:32:02] Time left 124.22 secs

[19:32:02] Layer 1 training completed.

[19:32:02] Blending: optimization starts with equal weights. Score = 0.8359385
[19:32:03] Blending: iteration 0: score = 0.8377845, weights = [0.06010976 0.12005781 0.16415095 0.15849096 0.49719054]
[19:32:04] Blending: iteration 1: score = 0.8378594, weights = [0.05726327 0.14433758 0.17033432 0.06575598 0.5623088 ]
[19:32:04] Blending: iteration 2: score = 0.8378623, weights = [0.05531821 0.14589803 0.17041638 0.06578765 0.5625797 ]
[19:32:05] Blending: no improvements for score. Terminated.

[19:32:05] Blending: best score = 0.8378623, best weights = [0.05531821 0.14589803 0.17041638 0.06578765 0.5625797 ]
[19:32:05] Automl preset training completed in 478.28 seconds

[19:32:05] Model description:
Final prediction for new objects (level 0) = 
	 

In [ ]:
# Конфиг 2: только градиентный бустинг
automl_2 = TabularAutoML(
    task=task,
    timeout=600,
    cpu_limit=4,
    general_params={'use_algos': [['lgb', 'lgb_tuned', 'cb']]},
    reader_params={'n_jobs': 4, 'cv': 5, 'random_state': RANDOM_STATE}
)

oof_pred_2 = automl_2.fit_predict(train_data, roles=roles, verbose=1)
val_pred_2 = automl_2.predict(X_val)
score_2 = roc_auc_score(y_val, val_pred_2.data[:, 0])

[19:32:29] Stdout logging level is INFO.
[19:32:29] Task: binary

[19:32:29] Start automl preset with listed constraints:
[19:32:29] - time: 600.00 seconds
[19:32:29] - CPU: 4 cores
[19:32:29] - memory: 16 GB

[19:32:29] Train data shape: (60816, 371)

[19:32:43] Layer 1 train process start. Time left 585.87 secs
[19:32:44] Selector_LightGBM fitting and predicting completed
[19:32:46] Start fitting Lvl_0_Pipe_0_Mod_0_LightGBM ...
[19:32:55] Fitting Lvl_0_Pipe_0_Mod_0_LightGBM finished. score = 0.8310780537385636
[19:32:55] Lvl_0_Pipe_0_Mod_0_LightGBM fitting and predicting completed
[19:32:55] Start hyperparameters optimization for Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM ... Time budget is 227.94 secs


Optimization Progress: 100%|██████████| 101/101 [03:24<00:00,  2.02s/it, best_trial=90, best_value=0.848]

[19:36:19] Hyperparameters optimization for Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM completed
[19:36:19] Start fitting Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM ...


[19:36:25] Fitting Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM finished. score = 0.8355422613072978
[19:36:25] Lvl_0_Pipe_0_Mod_1_Tuned_LightGBM fitting and predicting completed
[19:36:25] Start fitting Lvl_0_Pipe_0_Mod_2_CatBoost ...
[19:36:43] Fitting Lvl_0_Pipe_0_Mod_2_CatBoost finished. score = 0.8327864639035865
[19:36:43] Lvl_0_Pipe_0_Mod_2_CatBoost fitting and predicting completed
[19:36:43] Time left 345.13 secs

[19:36:43] Layer 1 training completed.

[19:36:43] Blending: optimization starts with equal weights. Score = 0.8369573
[19:36:44] Blending: iteration 0: score = 0.8374478, weights = [0.06811816 0.53266585 0.399216  ]
[19:36:44] Blending: no improvements for score. Terminated.

[19:36:44] Blending: best score = 0.8374478, best weights = [0.06811816 0.53266585 0.399216  ]
[19:36:44] Automl preset training completed in 255.40 seconds

[19:36:44] Model description:
Final prediction for new objects (level 0) = 
	 0.06812 * (5 averaged models Lvl_0_Pipe_0_Mod_0_LightGBM) +
	 0.53267 *

In [ ]:
# Выбор лучшей LAMA модели и сохранение предсказаний
best_lama = automl_1 if score_1 > score_2 else automl_2
best_lama_model = "LAMA Конфиг 1" if score_1 > score_2 else "LAMA Конфиг 2"
print(f"Лучшая LAMA модель: {best_lama_model} с качеством ROC AUC = {max(score_1, score_2):.6f}")

test_pred_lama = best_lama.predict(df_test).data[:, 0]
submission_lama = pd.DataFrame({'ID': df_test.ID, 'TARGET': test_pred_lama})
submission_lama.to_csv('submission_lama.csv', index=False)

Лучшая LAMA модель: LAMA Конфиг 2 с качеством ROC AUC = 0.848186


## XGBoost с Random Search

Случайный поиск гиперпараметров с 10-fold кросс-валидацией и early stopping.

In [ ]:
from xgboost import XGBClassifier

# Подготовка данных для XGBoost
X_xgb = X.copy()
y_xgb = y.copy()

results = []
n_trials = 20
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

# Random search по гиперпараметрам
for i in tqdm(range(n_trials), desc='XGBoost Random Search'):
    # Случайные гиперпараметры
    colsample_bytree = np.random.uniform(0.5, 0.99)
    subsample = np.random.uniform(0.7, 0.99)
    learning_rate = np.random.uniform(0.005, 0.05)
    max_depth = np.random.randint(4, 9)
    max_delta_step = np.random.randint(0, 4)
    base_score = np.random.uniform(0.05, 0.5)
    
    cv_scores = []
    best_rounds = []
    
    # Кросс-валидация
    for train_idx, val_idx in cv.split(X_xgb, y_xgb):
        X_tr, X_vl = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
        y_tr, y_vl = y_xgb.iloc[train_idx], y_xgb.iloc[val_idx]
        
        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            colsample_bytree=colsample_bytree,
            subsample=subsample,
            learning_rate=learning_rate,
            max_depth=max_depth,
            max_delta_step=max_delta_step,
            base_score=base_score,
            n_estimators=10000,
            early_stopping_rounds=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        )
        
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        pred = model.predict_proba(X_vl)[:, 1]
        cv_scores.append(roc_auc_score(y_vl, pred))
    
    best_auc = np.mean(cv_scores)
    
    results.append({
        'trial': i + 1,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'max_delta_step': max_delta_step,
        'base_score': base_score,
        'best_auc': best_auc
    })

XGBoost Random Search: 100%|██████████| 20/20 [35:20<00:00, 106.00s/it]


In [ ]:
# Сохранение результатов поиска
results_df = pd.DataFrame(results)
results_df.to_csv('xgb_random_search_results.csv', index=False)

# Лучшие гиперпараметры
best_trial = results_df.loc[results_df['best_auc'].idxmax()]

# Финальная модель на всех данных
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    colsample_bytree=best_trial['colsample_bytree'],
    subsample=best_trial['subsample'],
    learning_rate=best_trial['learning_rate'],
    max_depth=int(best_trial['max_depth']),
    max_delta_step=int(best_trial['max_delta_step']),
    base_score=best_trial['base_score'],
    n_estimators=1000,
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=0
)

xgb_model.fit(X_xgb, y_xgb)

# Оценка на валидации
pred_val_xgb = xgb_model.predict_proba(X_val)[:, 1]
score_xgb = roc_auc_score(y_val, pred_val_xgb)

Видим, что победили LAMA по ROC AUC на валидации

In [ ]:
# Сравнение результатов
results_comparison = pd.DataFrame({
    'Model': ['LAMA Config 1', 'LAMA Config 2', 'XGBoost'],
    'ROC AUC': [score_1, score_2, score_xgb]
}).sort_values('ROC AUC', ascending=False)

results_comparison

,Model,ROC AUC
2,XGBoost,0.888849
1,LAMA Config 2,0.848186
0,LAMA Config 1,0.847196


In [ ]:
# Предсказания XGBoost на тесте
X_test_xgb = df_test.drop(columns=['ID'])
test_ids = df_test['ID'].copy()

pred_test_xgb = xgb_model.predict_proba(X_test_xgb)[:, 1]

submission_xgb = pd.DataFrame({'ID': test_ids, 'TARGET': pred_test_xgb})
submission_xgb.to_csv('submission_xgb.csv', index=False)

## Финальные результаты

Видим, что победили LAMA по private и public оценке

![Результаты LAMA](kaggle_score_lama.png)

![Результаты xgb](kaggle_score_xgb.png)